# パスワードラボ

「あと1文字長ければ、探す時間はどれくらい変わる？」

実験用の文字列を用意して、探し方と計算方法を比べてみましょう。**実際に使っているパスワードは入力しないでください。**

セル1から5まで順に実行します。まずは数字4文字のデモで試せます。GPUがなくても動きます。

[詳しい使い方](https://github.com/shogo-ishikawa/2026_3Q_M3Th3/blob/main/projects/password-lab/docs/manual.md)

In [ ]:
#@title 1. 実行環境を準備する
gpu_enabled = True #@param {type:"boolean"}
import importlib.util
import subprocess
import sys
import tempfile
from pathlib import Path
from html import escape
from IPython.display import display, HTML

missing = [name for name in ("numpy", "numba") if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *missing], check=True)

# GPUがある環境で、CuPyだけが未導入なら追加します。
import shutil
if gpu_enabled and shutil.which("nvidia-smi") and importlib.util.find_spec("cupy") is None:
    try:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "cupy-cuda12x"], check=True)
    except subprocess.CalledProcessError:
        print("GPU用ライブラリの準備ができませんでした。CPUで続けられます。")

runtime_dir = Path(tempfile.mkdtemp(prefix="password-lab-"))
(runtime_dir / "hash_engines.py").write_text("\"\"\"教材用の固定長ASCII候補探索。暗号ライブラリとしては使用しない。\"\"\"\n\nimport hashlib\nimport struct\nfrom functools import lru_cache\nfrom numbers import Integral\n\nMAX_INDEX = 2**63 - 1\nMAX_BATCH = 65_536\n\n_K = (\n    0x428A2F98, 0x71374491, 0xB5C0FBCF, 0xE9B5DBA5,\n    0x3956C25B, 0x59F111F1, 0x923F82A4, 0xAB1C5ED5,\n    0xD807AA98, 0x12835B01, 0x243185BE, 0x550C7DC3,\n    0x72BE5D74, 0x80DEB1FE, 0x9BDC06A7, 0xC19BF174,\n    0xE49B69C1, 0xEFBE4786, 0x0FC19DC6, 0x240CA1CC,\n    0x2DE92C6F, 0x4A7484AA, 0x5CB0A9DC, 0x76F988DA,\n    0x983E5152, 0xA831C66D, 0xB00327C8, 0xBF597FC7,\n    0xC6E00BF3, 0xD5A79147, 0x06CA6351, 0x14292967,\n    0x27B70A85, 0x2E1B2138, 0x4D2C6DFC, 0x53380D13,\n    0x650A7354, 0x766A0ABB, 0x81C2C92E, 0x92722C85,\n    0xA2BFE8A1, 0xA81A664B, 0xC24B8B70, 0xC76C51A3,\n    0xD192E819, 0xD6990624, 0xF40E3585, 0x106AA070,\n    0x19A4C116, 0x1E376C08, 0x2748774C, 0x34B0BCB5,\n    0x391C0CB3, 0x4ED8AA4A, 0x5B9CCA4F, 0x682E6FF3,\n    0x748F82EE, 0x78A5636F, 0x84C87814, 0x8CC70208,\n    0x90BEFFFA, 0xA4506CEB, 0xBEF9A3F7, 0xC67178F2,\n)\n_H = (\n    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,\n    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,\n)\n\n\ndef _integer(value, name):\n    if isinstance(value, bool) or not isinstance(value, Integral):\n        raise ValueError(f\"{name}は整数で指定してください。\")\n    return int(value)\n\n\ndef _settings(alphabet, length):\n    length = _integer(length, \"桁数\")\n    if not isinstance(alphabet, str) or not alphabet:\n        raise ValueError(\"使用文字を1文字以上指定してください。\")\n    if not alphabet.isascii() or len(set(alphabet)) != len(alphabet):\n        raise ValueError(\"使用文字は重複のないASCII文字にしてください。\")\n    if not 1 <= length <= 32:\n        raise ValueError(\"実探索の桁数は1〜32にしてください。\")\n    return length, len(alphabet)**length\n\n\ndef _decode(index, alphabet, length):\n    chars = [alphabet[0]] * length\n    base = len(alphabet)\n    for pos in range(length - 1, -1, -1):\n        index, digit = divmod(index, base)\n        chars[pos] = alphabet[digit]\n    return \"\".join(chars)\n\n\ndef candidate_from_index(index, alphabet, length):\n    length, total = _settings(alphabet, length)\n    index = _integer(index, \"候補番号\")\n    if not 0 <= index <= MAX_INDEX or index >= total:\n        raise ValueError(\"候補番号が探索可能な範囲外です。\")\n    return _decode(index, alphabet, length)\n\n\ndef _range(start, count, total):\n    start = _integer(start, \"開始番号\")\n    count = _integer(count, \"候補数\")\n    cap = min(total, MAX_INDEX + 1)\n    if start < 0 or count < 0 or count > MAX_BATCH or start + count > cap:\n        raise ValueError(\"探索範囲が不正です。1回の候補数は65536以下です。\")\n    return start, count\n\n\n@lru_cache(maxsize=1)\ndef _numba_kernels():\n    import numpy as np\n    from numba import njit, prange\n\n    constants = np.array(_K, dtype=np.uint32)\n    initial = np.array(_H, dtype=np.uint32)\n\n    @njit(inline=\"always\")\n    def rotate(x, shift):\n        x = np.uint32(x)\n        shift = np.uint32(shift)\n        return np.uint32(\n            (x >> shift) | (x << (np.uint32(32) - shift))\n        )\n\n    @njit\n    def matches(index, codes, length, target):\n        # SHA-256の1ブロック分を構築する。\n        w = np.zeros(64, dtype=np.uint32)\n        remainder = np.uint64(index)\n        base = np.uint64(len(codes))\n        for pos in range(length - 1, -1, -1):\n            digit = int(remainder % base)\n            byte = codes[digit]\n            shift = np.uint32(24 - 8 * (pos % 4))\n            w[pos // 4] = np.uint32(\n                w[pos // 4] | (np.uint32(byte) << shift)\n            )\n            remainder //= base\n        shift = np.uint32(24 - 8 * (length % 4))\n        w[length // 4] = np.uint32(\n            w[length // 4] | (np.uint32(0x80) << shift)\n        )\n        w[15] = np.uint32(length * 8)\n\n        for i in range(16, 64):\n            s0 = np.uint32(\n                rotate(w[i - 15], 7) ^ rotate(w[i - 15], 18)\n                ^ (w[i - 15] >> np.uint32(3))\n            )\n            s1 = np.uint32(\n                rotate(w[i - 2], 17) ^ rotate(w[i - 2], 19)\n                ^ (w[i - 2] >> np.uint32(10))\n            )\n            w[i] = np.uint32(w[i - 16] + s0 + w[i - 7] + s1)\n\n        a, b, c, d = initial[0], initial[1], initial[2], initial[3]\n        e, f, g, h = initial[4], initial[5], initial[6], initial[7]\n        for i in range(64):\n            upper = np.uint32(rotate(e, 6) ^ rotate(e, 11) ^ rotate(e, 25))\n            choice = np.uint32((e & f) ^ ((~e) & g))\n            t1 = np.uint32(h + upper + choice + constants[i] + w[i])\n            lower = np.uint32(rotate(a, 2) ^ rotate(a, 13) ^ rotate(a, 22))\n            majority = np.uint32((a & b) ^ (a & c) ^ (b & c))\n            t2 = np.uint32(lower + majority)\n            h, g, f, e = g, f, e, np.uint32(d + t1)\n            d, c, b, a = c, b, a, np.uint32(t1 + t2)\n\n        return (\n            np.uint32(a + initial[0]) == target[0]\n            and np.uint32(b + initial[1]) == target[1]\n            and np.uint32(c + initial[2]) == target[2]\n            and np.uint32(d + initial[3]) == target[3]\n            and np.uint32(e + initial[4]) == target[4]\n            and np.uint32(f + initial[5]) == target[5]\n            and np.uint32(g + initial[6]) == target[6]\n            and np.uint32(h + initial[7]) == target[7]\n        )\n\n    @njit\n    def serial(start, count, codes, length, target):\n        found = np.int64(-1)\n        for offset in range(count):\n            index = np.uint64(start) + np.uint64(offset)\n            hit = matches(index, codes, length, target)\n            # 比較のため、一致後もバッチ全体を処理する。\n            if hit and found < 0:\n                found = np.int64(index)\n        return found\n\n    @njit(parallel=True)\n    def parallel(start, count, codes, length, target):\n        hits = np.full(count, -1, dtype=np.int64)\n        for offset in prange(count):\n            index = np.uint64(start) + np.uint64(offset)\n            if matches(index, codes, length, target):\n                hits[offset] = np.int64(index)\n        for offset in range(count):\n            if hits[offset] >= 0:\n                return hits[offset]\n        return np.int64(-1)\n\n    return serial, parallel\n\n\n_CUDA = r\"\"\"\n__device__ __constant__ unsigned int K[64] = { __K_VALUES__ };\n\n__device__ __forceinline__\nunsigned int rr(unsigned int x, unsigned int n) {\n    return (x >> n) | (x << (32 - n));\n}\n\n__device__\nbool sha_match(\n    unsigned long long index, const unsigned char* codes,\n    unsigned int base, unsigned int length, const unsigned int* target\n) {\n    unsigned int w[64] = {0};\n    for (int pos = (int)length - 1; pos >= 0; --pos) {\n        unsigned int digit = (unsigned int)(index % base);\n        w[pos / 4] |= ((unsigned int)codes[digit]) << (24 - 8 * (pos % 4));\n        index /= base;\n    }\n    w[length / 4] |= 0x80u << (24 - 8 * (length % 4));\n    w[15] = length * 8;\n    for (int i = 16; i < 64; ++i) {\n        unsigned int s0 = rr(w[i-15],7) ^ rr(w[i-15],18) ^ (w[i-15] >> 3);\n        unsigned int s1 = rr(w[i-2],17) ^ rr(w[i-2],19) ^ (w[i-2] >> 10);\n        w[i] = w[i-16] + s0 + w[i-7] + s1;\n    }\n    unsigned int a=0x6a09e667u, b=0xbb67ae85u;\n    unsigned int c=0x3c6ef372u, d=0xa54ff53au;\n    unsigned int e=0x510e527fu, f=0x9b05688cu;\n    unsigned int g=0x1f83d9abu, h=0x5be0cd19u;\n    for (int i = 0; i < 64; ++i) {\n        unsigned int s1 = rr(e,6) ^ rr(e,11) ^ rr(e,25);\n        unsigned int ch = (e & f) ^ ((~e) & g);\n        unsigned int t1 = h + s1 + ch + K[i] + w[i];\n        unsigned int s0 = rr(a,2) ^ rr(a,13) ^ rr(a,22);\n        unsigned int maj = (a & b) ^ (a & c) ^ (b & c);\n        unsigned int t2 = s0 + maj;\n        h=g; g=f; f=e; e=d+t1; d=c; c=b; b=a; a=t1+t2;\n    }\n    return a+0x6a09e667u==target[0] && b+0xbb67ae85u==target[1]\n        && c+0x3c6ef372u==target[2] && d+0xa54ff53au==target[3]\n        && e+0x510e527fu==target[4] && f+0x9b05688cu==target[5]\n        && g+0x1f83d9abu==target[6] && h+0x5be0cd19u==target[7];\n}\n\nextern \"C\" __global__\nvoid search_sha256(\n    unsigned long long start, unsigned int count,\n    const unsigned char* codes, unsigned int base, unsigned int length,\n    const unsigned int* target, unsigned long long* result\n) {\n    unsigned int offset = blockIdx.x * blockDim.x + threadIdx.x;\n    if (offset >= count) return;\n    unsigned long long index = start + (unsigned long long)offset;\n    if (sha_match(index, codes, base, length, target))\n        atomicMin(result, index);\n}\n\"\"\"\n\n\ndef make_engine(name, alphabet, length, target_digest_bytes):\n    \"\"\"全バッチを検証し、最小の一致indexまたはNoneを返す関数を作る。\"\"\"\n    length, total = _settings(alphabet, length)\n    if not isinstance(target_digest_bytes, (bytes, bytearray, memoryview)):\n        raise ValueError(\"照合値はSHA-256の32バイトで指定してください。\")\n    digest = bytes(target_digest_bytes)\n    if len(digest) != 32:\n        raise ValueError(\"照合値はSHA-256の32バイトで指定してください。\")\n    if name not in (\"Python\", \"Numba\", \"Numba parallel\", \"GPU\"):\n        raise ValueError(\"計算方法が不明です。\")\n\n    if name == \"Python\":\n        def check(start, count):\n            start, count = _range(start, count, total)\n            found = None\n            for index in range(start, start + count):\n                candidate = _decode(index, alphabet, length).encode(\"ascii\")\n                hit = hashlib.sha256(candidate).digest() == digest\n                if hit and found is None:\n                    found = index\n            return found\n        return check\n\n    import numpy as np\n    codes = np.frombuffer(alphabet.encode(\"ascii\"), dtype=np.uint8).copy()\n    target = np.array(struct.unpack(\">8I\", digest), dtype=np.uint32)\n\n    if name in (\"Numba\", \"Numba parallel\"):\n        serial, parallel = _numba_kernels()\n        kernel = parallel if name == \"Numba parallel\" else serial\n\n        def check(start, count):\n            start, count = _range(start, count, total)\n            if count == 0:\n                return None\n            found = int(kernel(\n                np.int64(start), np.int64(count), codes, length, target\n            ))\n            return None if found < 0 else found\n        return check\n\n    import cupy as cp\n    if cp.cuda.runtime.getDeviceCount() == 0:\n        raise RuntimeError(\"GPUが見つかりません。\")\n    source = _CUDA.replace(\n        \"__K_VALUES__\", \", \".join(f\"0x{k:08x}u\" for k in _K)\n    )\n    kernel = cp.RawKernel(source, \"search_sha256\")\n    device_codes, device_target = cp.asarray(codes), cp.asarray(target)\n    sentinel = 2**64 - 1\n\n    def check(start, count):\n        start, count = _range(start, count, total)\n        if count == 0:\n            return None\n        result = cp.full(1, sentinel, dtype=cp.uint64)\n        kernel(\n            ((count + 255) // 256,), (256,),\n            (\n                np.uint64(start), np.uint32(count), device_codes,\n                np.uint32(len(alphabet)), np.uint32(length),\n                device_target, result,\n            ),\n        )\n        cp.cuda.get_current_stream().synchronize()\n        found = int(cp.asnumpy(result)[0])\n        return None if found == sentinel else found\n    return check\n", encoding="utf-8")
(runtime_dir / "password_core.py").write_text("\"\"\"実験用パスワードだけを対象に、探索順序と計算速度を比べます。\"\"\"\nimport hashlib\nimport itertools\nimport math\nimport secrets\nimport string\nimport time\n\nfrom hash_engines import make_engine, candidate_from_index\n\nCHARSETS = {\n    \"digits\": string.digits,\n    \"lower\": string.ascii_lowercase,\n    \"upper\": string.ascii_uppercase,\n    \"symbols\": string.punctuation,\n}\nWORDS = [\n    \"password\", \"123456\", \"1234\", \"0000\", \"1111\", \"qwerty\",\n    \"abc123\", \"welcome\", \"letmein\", \"sakura\", \"coffee\", \"python\",\n    \"orange\", \"summer\", \"winter\", \"music\", \"hello\", \"dragon\",\n]\nMAX_BATCH = 8192\nVERSION = \"1.2.0\"\nALL_CHARACTERS = \"\".join(CHARSETS.values())\nGROUPS = {\n    \"数字\": CHARSETS[\"digits\"], \"小文字\": CHARSETS[\"lower\"],\n    \"大文字\": CHARSETS[\"upper\"], \"英字\": string.ascii_letters,\n    \"英数字\": string.ascii_letters + string.digits,\n    \"記号\": CHARSETS[\"symbols\"], \"すべて\": ALL_CHARACTERS, \"なし\": \"\",\n}\n\n\ndef alphabet_for(digits=True, lower=True, upper=True, symbols=False):\n    flags = [digits, lower, upper, symbols]\n    alphabet = \"\".join(value for value, use in zip(CHARSETS.values(), flags) if use)\n    if not alphabet:\n        raise ValueError(\"使える文字を1種類以上選んでください。\")\n    return alphabet\n\n\ndef validate_password(password):\n    if not isinstance(password, str) or not 1 <= len(password) <= 32:\n        raise ValueError(\"実験用の文字列を1〜32文字で指定してください。\")\n    if any(ord(c) < 33 or ord(c) > 126 for c in password):\n        raise ValueError(\"今回は半角の英数字・記号（空白なし）を使ってください。\")\n    return password\n\n\ndef generate_password(length, alphabet):\n    if not isinstance(length, int) or not 1 <= length <= 32:\n        raise ValueError(\"長さは1〜32文字にしてください。\")\n    if not alphabet or len(set(alphabet)) != len(alphabet):\n        raise ValueError(\"文字集合を確認してください。\")\n    return \"\".join(secrets.choice(alphabet) for _ in range(length))\n\n\ndef dictionary_candidates():\n    yield from WORDS\n\n\ndef rule_candidates():\n    # この小さな教材用リストは、実際の出現頻度を再現するものではありません。\n    suffixes = [\"\", \"1\", \"12\", \"123\", \"2024\", \"2025\", \"2026\", \"2027\", \"!\"]\n    for word in WORDS:\n        for base in dict.fromkeys([word, word.capitalize(), word.upper()]):\n            for suffix in suffixes:\n                yield base + suffix\n                if suffix and suffix != \"!\":\n                    yield base + suffix + \"!\"\n\n\ndef _length(value, label):\n    if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= 32:\n        raise ValueError(f\"{label}は1〜32の整数にしてください。\")\n    return value\n\n\ndef make_attack(knowledge=\"手がかりなし\", *, digits=True, lower=False,\n                upper=False, symbols=False, length_known=True, search_length=4,\n                search_max_length=6, prefix=\"\", suffix=\"\", group1=\"数字\",\n                group1_count=4, group2=\"なし\", group2_count=0):\n    \"\"\"攻撃側の設定だけで候補範囲を作る。正解の情報は受け取らない。\"\"\"\n    if knowledge not in (\"手がかりなし\", \"文字の種類・文字数を指定\", \"文字の並び・一部を指定\"):\n        raise ValueError(\"手がかりの種類を選んでください。\")\n    if knowledge == \"文字の並び・一部を指定\":\n        for fixed in (prefix, suffix):\n            if not isinstance(fixed, str) or any(not 33 <= ord(c) <= 126 for c in fixed):\n                raise ValueError(\"固定する文字には半角の英数字・記号（空白なし）を使ってください。\")\n        positions = list(prefix)\n        parts = [prefix] if prefix else []\n        groups = []\n        for kind, count in ((group1, group1_count), (group2, group2_count)):\n            if kind not in GROUPS:\n                raise ValueError(\"文字グループと文字数を確認してください。\")\n            if kind == \"なし\":\n                continue\n            if isinstance(count, bool) or not isinstance(count, int):\n                raise ValueError(\"文字グループと文字数を確認してください。\")\n            if not 0 <= count <= 32:\n                raise ValueError(\"文字グループの文字数は0〜32にしてください。\")\n            if count:\n                positions.extend([GROUPS[kind]] * count)\n                parts.append(f\"{kind}{count}文字\")\n                groups.append({\"kind\": kind, \"count\": count})\n        positions.extend(suffix)\n        _length(len(positions), \"並び全体の文字数\")\n        if suffix:\n            parts.append(suffix)\n        total = math.prod(len(chars) for chars in positions)\n        public = {\"knowledge\": knowledge, \"length_known\": True,\n                  \"min_length\": len(positions), \"max_length\": len(positions),\n                  \"prefix_length\": len(prefix), \"suffix_length\": len(suffix),\n                  \"groups\": groups, \"candidates\": total}\n        return {\"mode\": \"structure\", \"positions\": positions, \"candidates\": total,\n                \"min_length\": len(positions), \"max_length\": len(positions),\n                \"preview\": \"＋\".join(parts), \"public\": public}\n\n    if knowledge == \"手がかりなし\":\n        alphabet = ALL_CHARACTERS\n        minimum, maximum = 1, _length(search_max_length, \"探索する最大文字数\")\n        types = list(CHARSETS)\n    else:\n        alphabet = alphabet_for(digits, lower, upper, symbols)\n        maximum = _length(search_length if length_known else search_max_length,\n                          \"探索する文字数\")\n        minimum = maximum if length_known else 1\n        types = [name for name, enabled in zip(CHARSETS, (digits, lower, upper, symbols)) if enabled]\n    total = sum(len(alphabet) ** length for length in range(minimum, maximum + 1))\n    length_text = f\"{maximum}文字\" if minimum == maximum else f\"1〜{maximum}文字を短い順\"\n    public = {\"knowledge\": knowledge, \"length_known\": minimum == maximum,\n              \"min_length\": minimum, \"max_length\": maximum,\n              \"charset_types\": types, \"alphabet_size\": len(alphabet), \"candidates\": total}\n    return {\"mode\": \"charset\", \"alphabet\": alphabet, \"min_length\": minimum,\n            \"max_length\": maximum, \"candidates\": total,\n            \"preview\": f\"{len(alphabet)}種類の文字・{length_text}\", \"public\": public}\n\n\ndef structure_candidates(positions):\n    for values in itertools.product(*positions):\n        yield \"\".join(values)\n\n\ndef candidate_allowed(candidate, attack):\n    if attack[\"mode\"] == \"structure\":\n        return (len(candidate) == len(attack[\"positions\"])\n                and all(c in chars for c, chars in zip(candidate, attack[\"positions\"])))\n    return (attack[\"min_length\"] <= len(candidate) <= attack[\"max_length\"]\n            and all(c in attack[\"alphabet\"] for c in candidate))\n\n\ndef search_candidates(name, candidates, digest, seconds=2.0, limit=100000, attack=None):\n    if not 0 < seconds <= 30 or not 1 <= limit <= 1000000:\n        raise ValueError(\"実験の時間・件数の上限を確認してください。\")\n    begin = time.perf_counter()\n    tested = 0\n    seen = set()\n    for candidate in candidates:\n        if tested >= limit or time.perf_counter() - begin >= seconds:\n            return result(name, \"打切り\", tested, begin)\n        if (attack is not None and not candidate_allowed(candidate, attack)) or candidate in seen:\n            continue\n        seen.add(candidate)\n        tested += 1\n        if hashlib.sha256(candidate.encode(\"ascii\")).digest() == digest:\n            return result(name, \"発見\", tested, begin)\n    return result(name, \"この候補群では未発見\", tested, begin)\n\n\ndef result(name, state, tested, begin, **extra):\n    elapsed = max(time.perf_counter() - begin, 1e-12)\n    return {\"method\": name, \"state\": state, \"tested\": tested,\n            \"seconds\": elapsed, \"rate\": tested / elapsed, **extra}\n\n\ndef benchmark(alphabet, length, digest, include_gpu=True, batch=8192):\n    \"\"\"各方式で同じ件数を3回全走査。準備時間と実測を分けます。\"\"\"\n    total = len(alphabet) ** length\n    count = min(batch, total)\n    names = [\"Python\", \"Numba\", \"Numba parallel\"] + ([\"GPU\"] if include_gpu else [])\n    rows, engines = [], {}\n    for name in names:\n        prepare = time.perf_counter()\n        try:\n            # 既知の正解と不正解を、hashlibの結果に対して実行時にも照合します。\n            sample_index = min(count - 1, 7)\n            sample = candidate_from_index(sample_index, alphabet, length)\n            check = make_engine(name, alphabet, length,\n                                hashlib.sha256(sample.encode(\"ascii\")).digest())\n            actual = check(0, min(count, 8))\n            if actual != sample_index or check(0, sample_index) is not None:\n                raise RuntimeError(\"既知の正解との一致確認に失敗しました。\")\n            engine = make_engine(name, alphabet, length, digest)\n            engine(0, min(count, 8))\n            preparation = time.perf_counter() - prepare\n            times = []\n            for _ in range(3):\n                begin = time.perf_counter()\n                engine(0, count)\n                times.append(max(time.perf_counter() - begin, 1e-12))\n            elapsed = sorted(times)[1]\n            rows.append({\"engine\": name, \"state\": \"実測\", \"count\": count,\n                         \"seconds\": elapsed, \"rate\": count / elapsed,\n                         \"prepare_seconds\": preparation})\n            engines[name] = engine\n        except Exception as exc:\n            rows.append({\"engine\": name, \"state\": \"利用不可\",\n                         \"reason\": str(exc).splitlines()[0][:200]})\n    available = [row for row in rows if row[\"state\"] == \"実測\"]\n    if not available:\n        raise RuntimeError(\"利用できる照合方式がありません。\")\n    best = max(available, key=lambda row: row[\"rate\"])\n    return rows, best[\"engine\"], engines[best[\"engine\"]]\n\n\ndef brute_search(attack, digest, seconds=2.0, limit=100000, engine_name=\"Python\"):\n    \"\"\"長さをまたいでも、1つの時間・件数上限を共有する。\"\"\"\n    if not 0 < seconds <= 30 or not 1 <= limit <= 1000000:\n        raise ValueError(\"実験の時間・件数の上限を確認してください。\")\n    begin = time.perf_counter()\n    tested = 0\n    alphabet = attack[\"alphabet\"]\n    for length in range(attack[\"min_length\"], attack[\"max_length\"] + 1):\n        if tested >= limit or time.perf_counter() - begin >= seconds:\n            return result(\"総当たり\", \"打切り\", tested, begin, engine=engine_name)\n        engine = make_engine(engine_name, alphabet, length, digest)\n        total, index = len(alphabet) ** length, 0\n        while index < total:\n            if (tested >= limit or time.perf_counter() - begin >= seconds or index >= 2**63):\n                return result(\"総当たり\", \"打切り\", tested, begin, engine=engine_name)\n            count = min(MAX_BATCH, total - index, limit - tested, 2**63 - index)\n            match = engine(index, count)\n            index += count\n            tested += count\n            if match is not None:\n                return result(\"総当たり\", \"発見\", tested, begin, engine=engine_name)\n    return result(\"総当たり\", \"全候補で未発見\", tested, begin, engine=engine_name)\n\n\ndef estimate(alphabet, length, rate, min_length=None):\n    if not math.isfinite(rate) or rate <= 0:\n        raise ValueError(\"実測速度が必要です。\")\n    minimum = length if min_length is None else min_length\n    total = sum(len(alphabet) ** n for n in range(minimum, length + 1))\n    return {\"candidates\": total, \"average_seconds\": (total + 1) / (2 * rate),\n            \"exhaustive_seconds\": total / rate,\n            \"assumption\": \"指定した全候補が同じ確率で選ばれ、全長で同じSHA-256照合速度と仮定\"}\n\n\ndef benchmark_structure(positions, digest, batch=8192):\n    \"\"\"位置ごとに使える文字が異なる候補生成も含め、別に測定する。\"\"\"\n    count = min(batch, math.prod(len(chars) for chars in positions))\n    def scan():\n        for candidate in itertools.islice(structure_candidates(positions), count):\n            hashlib.sha256(candidate.encode(\"ascii\")).digest() == digest\n    scan()\n    times = []\n    for _ in range(3):\n        begin = time.perf_counter()\n        scan()\n        times.append(max(time.perf_counter() - begin, 1e-12))\n    elapsed = sorted(times)[1]\n    return [{\"engine\": \"Python hashlib（並びの指定）\", \"state\": \"実測\",\n             \"count\": count, \"seconds\": elapsed, \"rate\": count / elapsed,\n             \"prepare_seconds\": 0.0}]\n\n\ndef duration(seconds):\n    if seconds < 1:\n        return f\"{seconds * 1000:.2f}ミリ秒\"\n    if seconds < 60:\n        return f\"{seconds:.2f}秒\"\n    if seconds < 3600:\n        return f\"{seconds / 60:.1f}分\"\n    if seconds < 86400:\n        return f\"{seconds / 3600:.1f}時間\"\n    if seconds < 31557600:\n        return f\"{seconds / 86400:.1f}日\"\n    years = seconds / 31557600\n    return f\"{years:,.1f}年\" if years < 1e9 else f\"{years:.2e}年\"\n\n\ndef run_experiment(password, attack=None,\n                   seconds=2.0, limit=100000, include_gpu=True):\n    validate_password(password)\n    if not 0 < seconds <= 30 or not 1 <= limit <= 1000000:\n        raise ValueError(\"時間は30秒以内、件数は100万件以内にしてください。\")\n    if attack is None:\n        attack = make_attack()\n    digest = hashlib.sha256(password.encode(\"ascii\")).digest()\n    if attack[\"mode\"] == \"structure\":\n        print(\"指定した並びの候補生成とSHA-256照合を測っています…\")\n        speeds = benchmark_structure(attack[\"positions\"], digest)\n        fastest = speeds[0][\"engine\"]\n        search_row = search_candidates(\"指定した並び\", structure_candidates(attack[\"positions\"]),\n                                       digest, seconds, limit)\n        rate = speeds[0][\"rate\"]\n        total = attack[\"candidates\"]\n        estimated = {\"candidates\": total, \"average_seconds\": (total + 1) / (2 * rate),\n                     \"exhaustive_seconds\": total / rate,\n                     \"assumption\": \"指定した並びの全候補が同じ確率で選ばれると仮定\"}\n    else:\n        print(\"攻撃側で指定した最大文字数で、同じSHA-256照合の速さを測っています…\")\n        speeds, fastest, _ = benchmark(attack[\"alphabet\"], attack[\"max_length\"],\n                                      digest, include_gpu)\n        search_row = brute_search(attack, digest, seconds, limit, fastest)\n        rate = next(r[\"rate\"] for r in speeds if r[\"engine\"] == fastest)\n        estimated = estimate(attack[\"alphabet\"], attack[\"max_length\"], rate, attack[\"min_length\"])\n    trials = [\n        search_candidates(\"辞書\", dictionary_candidates(), digest, seconds, limit, attack),\n        search_candidates(\"単語の変形\", rule_candidates(), digest, seconds, limit, attack),\n        search_row,\n    ]\n    return {\n        \"version\": VERSION,\n        \"verification\": \"SHA-256（教材用・実サービスの保存方式を再現しません）\",\n        \"attack\": attack[\"public\"],\n        \"fastest_engine\": fastest, \"benchmarks\": speeds, \"trials\": trials,\n        \"estimate\": estimated,\n        \"notes\": [\n            \"探索の打切りは安全の証明ではありません。\",\n            \"平均時間は指定した全候補が同じ確率で選ばれる仮定です。人が考えた文字列の強さではありません。\",\n            \"攻撃側の手がかりはセル3で指定し、正解の長さや文字種から取得しません。\",\n            \"辞書と単語の変形も、攻撃側で指定した候補範囲の中で試します。\",\n            \"文字数が不明なら短い順に調べ、時間・件数の上限は各方法の全体で共有します。\",\n            \"最大文字数で測定した代表速度を、短い文字数にも同じと仮定して推定します。\"\n            if attack[\"mode\"] != \"structure\" else\n            \"並び指定では位置ごとに文字集合が異なるため、Python hashlibで別に速度を測定します。\",\n            \"実サービスの試行制限、保存方式、フィッシングなどはこの実験の対象外です。\",\n            \"速度は同じ候補範囲を3回全走査した中央値です。初回準備時間は別表示です。\",\n            \"バッチは途中で止めないため、総当たりは発見後の余分な候補も件数に含みます。\",\n        ],\n    }\n\n\ndef pbkdf2_comparison(password, iterations=100000, repeats=3):\n    \"\"\"保存方式で照合コストが変わることを、同じ文字列で比較します。\"\"\"\n    validate_password(password)\n    if iterations < 10000 or iterations > 1000000:\n        raise ValueError(\"反復回数は1万〜100万回にしてください。\")\n    raw, salt = password.encode(\"ascii\"), secrets.token_bytes(16)\n    rows = []\n    for name, function in [\n        (\"SHA-256\", lambda: hashlib.sha256(raw).digest()),\n        (f\"PBKDF2-HMAC-SHA256 {iterations:,}回\",\n         lambda: hashlib.pbkdf2_hmac(\"sha256\", raw, salt, iterations)),\n    ]:\n        function()\n        begin = time.perf_counter()\n        for _ in range(repeats):\n            function()\n        elapsed = (time.perf_counter() - begin) / repeats\n        rows.append({\"verification\": name, \"seconds_per_check\": elapsed})\n    return rows\n", encoding="utf-8")
sys.path.insert(0, str(runtime_dir))
for name in ("hash_engines", "password_core"):
    sys.modules.pop(name, None)
from password_core import (
    alphabet_for, generate_password, validate_password, run_experiment,
    duration, pbkdf2_comparison, make_attack,
)

def show_table(rows, columns):
    header = "".join("<th>" + escape(label) + "</th>" for key, label in columns)
    body = ""
    for row in rows:
        values = []
        for key, label in columns:
            value = row.get(key, "—")
            if isinstance(value, float):
                value = f"{value:,.6f}" if "seconds" in key else f"{value:,.0f}"
            values.append("<td>" + escape(str(value)) + "</td>")
        body += "<tr>" + "".join(values) + "</tr>"
    display(HTML("<div style='overflow:auto'><table class='pw-table'><thead><tr>"
                 + header + "</tr></thead><tbody>" + body + "</tbody></table></div>"))

display(HTML("<style>.pw-table{border-collapse:collapse;width:100%;margin:12px 0}"
             ".pw-table td,.pw-table th{text-align:left;padding:10px;border-bottom:1px solid #ccd7df}"
             ".pw-table th{background:#edf4fa}.pw-card{padding:18px;border:1px solid #ccd7df;"
             "border-radius:10px;margin:12px 0}</style>"))
experiment = None
password = None
generation_alphabet = None
attack = None
print("準備できました。セル2へ進みましょう。初めてのNumba計測では変換処理に少し時間がかかります。")


## 実験用の文字列を用意しましょう

デモを選ぶか、自動生成・入力で練習用の文字列を用意します。ここで選ぶ文字種は**生成用**です。探す側が何を知っているかは、次のセルで別に決めます。

自動生成では、選んだ文字を合わせた集合から1文字ずつランダムに選びます。各文字種が必ず全部入るわけではありません。今回は半角の英数字・記号（空白なし）、1〜32文字を扱います。

In [ ]:
#@title 2. 実験用の文字列を用意する
source_mode = "デモ：数字4桁" #@param ["デモ：数字4桁", "デモ：単語＋数字", "自動生成", "自分で入力"]
password_length = 4 #@param {type:"integer"}
use_digits = True #@param {type:"boolean"}
use_lower = True #@param {type:"boolean"}
use_upper = True #@param {type:"boolean"}
use_symbols = False #@param {type:"boolean"}

experiment = None
storage_comparison = None
password = None
generation_alphabet = None
if source_mode == "デモ：数字4桁":
    password = "4729"
elif source_mode == "デモ：単語＋数字":
    password = "sakura2026"
elif source_mode == "自動生成":
    generation_alphabet = alphabet_for(use_digits, use_lower, use_upper, use_symbols)
    password = generate_password(password_length, generation_alphabet)
else:
    from getpass import getpass
    password = getpass("実験用の文字列：")
validate_password(password)
if source_mode != "自分で入力":
    print("今回の実験用文字列：", password)
print("文字列を用意できました。セル3で、探す側の手がかりを選びましょう。")


## 探す側は何を知っていますか？

- **手がかりなし**：半角の英数字・記号94種類を使い、1文字から指定した最大文字数まで短い順に探します。
- **文字の種類・文字数を指定**：「数字だけ」「4文字と分かっている」など、知っている条件を選びます。文字数が不明なら、`length_known`を外して最大文字数を指定します。
- **文字の並び・一部を指定**：固定する先頭・末尾と、文字グループの種類・文字数を組み合わせます。例えば先頭を`sakura`、グループ1を「数字・4文字」にすると「sakura＋数字4文字」です。

セル2の生成設定や正解を見て、探す範囲を自動で狭めることはありません。辞書と単語の変形も、ここで指定した範囲の中で試します。各方法は時間か件数の上限で止めます。**未発見は安全の証明ではありません。**

In [ ]:
#@title 3. 探す側の手がかりを選び、実際に試す
attack_knowledge = "手がかりなし" #@param ["手がかりなし", "文字の種類・文字数を指定", "文字の並び・一部を指定"]
#@markdown 「手がかりなし」は最大文字数だけを使います。
search_max_length = 6 #@param {type:"integer"}
#@markdown 「文字の種類・文字数を指定」で使う条件
attack_digits = True #@param {type:"boolean"}
attack_lower = False #@param {type:"boolean"}
attack_upper = False #@param {type:"boolean"}
attack_symbols = False #@param {type:"boolean"}
length_known = True #@param {type:"boolean"}
search_length = 4 #@param {type:"integer"}
#@markdown 「文字の並び・一部を指定」で使う条件。例：先頭 sakura ＋ 数字4文字
fixed_prefix = "" #@param {type:"string"}
character_group_1 = "数字" #@param ["数字", "小文字", "大文字", "英字", "英数字", "記号", "すべて", "なし"]
group_1_count = 4 #@param {type:"integer"}
character_group_2 = "なし" #@param ["数字", "小文字", "大文字", "英字", "英数字", "記号", "すべて", "なし"]
group_2_count = 0 #@param {type:"integer"}
fixed_suffix = "" #@param {type:"string"}
seconds_per_method = 2 #@param {type:"slider", min:1, max:10, step:1}
candidate_limit = 100000 #@param {type:"integer"}

experiment = None
storage_comparison = None
attack = make_attack(attack_knowledge,
    digits=attack_digits, lower=attack_lower, upper=attack_upper, symbols=attack_symbols,
    length_known=length_known, search_length=search_length, search_max_length=search_max_length,
    prefix=fixed_prefix, suffix=fixed_suffix,
    group1=character_group_1, group1_count=group_1_count,
    group2=character_group_2, group2_count=group_2_count)
print("今回の探す範囲：", attack["preview"])
print(f"候補の組み合わせ：{attack['candidates']:,}通り")
experiment = run_experiment(password, attack,
    seconds=float(seconds_per_method), limit=int(candidate_limit), include_gpu=gpu_enabled)

show_table(experiment["trials"], [
    ("method", "探し方"), ("state", "結果"), ("tested", "調べた件数"),
    ("seconds", "時間（秒）"), ("rate", "件／秒"), ("engine", "実行方式"),
])
found_rows = [r for r in experiment["trials"] if r["state"] == "発見"]
if found_rows:
    quickest = min(found_rows, key=lambda row: row["seconds"])
    print(f"今回は「{quickest['method']}」で最も短時間に見つかりました。")
    if any(r["method"] in ("辞書", "単語の変形") for r in found_rows):
        print("単語やよくある規則で予想できる文字列でした。長さだけでは強さを判断できない例です。")
else:
    print("この実験では未発見でした。探す条件が合っていない場合もあり、安全とは判断できません。")

print("\n同じ候補を調べる速さ（同じ件数を3回測定した中央値）")
show_table(experiment["benchmarks"], [
    ("engine", "実行方式"), ("state", "状態"), ("count", "候補数"),
    ("seconds", "時間（秒）"), ("rate", "件／秒"),
    ("prepare_seconds", "初回準備（秒）"), ("reason", "利用できない理由"),
])
print("使った方式：", experiment["fastest_engine"])
if attack["mode"] == "structure":
    print("並び指定では位置ごとに使える文字が異なるため、Python hashlibで別に測定します。")
else:
    print("攻撃側の最大文字数で代表速度を測り、総当たりには最も速い方式を使います。")
    print("GPUを使う場合はセル1のgpu_enabledを有効にし、GPUランタイムで上から実行してください。")


## 候補全体を調べる時間を見積もりましょう

文字数が分からない場合は、1文字・2文字…の候補数を足します。例えば数字だけで1〜4文字なら、10＋100＋1,000＋10,000＝11,110通りです。並びを指定した場合は、各位置で使える文字の種類数を掛け合わせます。

平均時間は、**指定した全候補が同じ確率で選ばれる**と仮定した値です。文字数が異なる場合も、最大文字数で測定した速度と同じと仮定します。人が考えた文字列の強さや、実サービスの突破時間を表すものではありません。

SHA-256は速度比較のための教材条件です。実サービスでは、照合を意図的に重くする保存方式やログインの試行制限も使われます。

In [ ]:
#@title 4. 候補全体を調べる時間と、保存方式による違いを見る
pbkdf2_iterations = 100000 #@param {type:"integer"}
storage_comparison = None
if experiment is None:
    raise RuntimeError("先にセル3を実行してください。")
est = experiment["estimate"]
print(f"攻撃側が指定した候補数：{est['candidates']:,}通り")
print("全候補が同じ確率で選ばれると仮定した推定")
print("平均的に見つかるまで：", duration(est["average_seconds"]))
print("全候補を調べるまで：", duration(est["exhaustive_seconds"]))
print(est["assumption"])
print("条件が正解と合っているかは判定しません。指定した候補範囲についての見積もりです。")

storage_comparison = pbkdf2_comparison(password, int(pbkdf2_iterations))
print("\n同じ文字列を1回照合するための時間（別の実測）")
show_table(storage_comparison, [
    ("verification", "照合方式"), ("seconds_per_check", "1回あたり（秒）"),
])
print("PBKDF2は計算を繰り返し、1回の推測にかかる負担を増やします。")
print("ここで選んだ反復回数を、実サービス向けの推奨設定として示すものではありません。")


## 結果を持ち帰りましょう

CSVとJSONをZIPで保存できます。探す条件の種類・文字数・候補数と、集計結果を記録します。入力した文字列、照合用ハッシュ、固定した先頭・末尾、各位置の候補文字は含みません。

次は探す側の手がかりを1つだけ変えて、結果を比べてみましょう。Notebookの入力設定や画面出力は、結果のZIPとは別にコピーへ残ることがあります。

In [ ]:
#@title 5. 結果をZIPで保存する
if experiment is None:
    raise RuntimeError("先にセル3を実行してください。")
import csv
import io
import json
import zipfile

def results_zip_bytes(experiment, storage_comparison):
    payload = dict(experiment)
    payload["storage_comparison"] = storage_comparison
    output = io.BytesIO()
    with zipfile.ZipFile(output, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("results.json", json.dumps(payload, ensure_ascii=False, indent=2))
        text = io.StringIO(newline="")
        columns = ["method", "state", "tested", "seconds", "rate", "engine"]
        writer = csv.DictWriter(text, columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(experiment["trials"])
        archive.writestr("search_results.csv", text.getvalue().encode("utf-8-sig"))
        archive.writestr("README.txt",
            "パスワードラボ：実験結果\n\n"
            "実験用のSHA-256照合による結果です。実サービスの突破時間ではありません。\n"
            "平均時間は指定した全候補が同じ確率で選ばれると仮定しています。\n"
            "入力文字列・照合用ハッシュ・固定文字列・各位置の候補文字は含みません。\n")
    return output.getvalue()

output_path = Path("password_lab_results.zip")
output_path.write_bytes(results_zip_bytes(experiment, storage_comparison))
try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:
    print("保存しました：", output_path.resolve())
